In [32]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

from sklearn.model_selection import train_test_split, StratifiedKFold

from sklearn.preprocessing import StandardScaler, MinMaxScaler

from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier

from sklearn.model_selection import GridSearchCV

from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score

from sklearn.base import clone

import warnings
warnings.simplefilter("ignore")

In [33]:
X = pd.read_csv("data/train.csv")
y = X["Survived"]
X_test = pd.read_csv("data/test.csv")
y_test = pd.read_csv("data/gender_submission.csv")["Survived"]
ids = X_test["PassengerId"]

In [34]:
X.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [35]:
# # gender OneHotEncoding
# ohe = OneHotEncoder(sparse_output=False, drop='first')
# encoded = ohe.fit_transform(X[['Sex']])
# encoded_df = pd.DataFrame(encoded, columns=ohe.get_feature_names_out())
# X = X.drop(columns=['Sex'])
# X = pd.concat([X, encoded_df], axis=1)

# encoded = ohe.transform(X_test[['Sex']])
# encoded_df = pd.DataFrame(encoded, columns=ohe.get_feature_names_out())
# X_test = X_test.drop(columns=['Sex'])
# X_test = pd.concat([X_test, encoded_df], axis=1)

In [36]:
# feature engineering
X["family"] = X["SibSp"] + X["Parch"]
X = X.drop(["SibSp", "Parch"], axis=1)

X_test["family"] = X_test["SibSp"] + X_test["Parch"]
X_test = X_test.drop(["SibSp", "Parch"], axis=1)

In [37]:
# fill Age NaN values
imputer = SimpleImputer(strategy='median')
X[['Age', 'Fare']] = imputer.fit_transform(X[['Age', 'Fare']])

X_test[['Age', 'Fare']] = imputer.transform(X_test[['Age', 'Fare']])

In [38]:
# Cabin
X["haveCabin"] = np.where(X["Cabin"].isnull(), 0, 1)
X = X.drop(["Cabin"], axis=1)

X_test["haveCabin"] = np.where(X_test["Cabin"].isnull(), 0, 1)
X_test = X_test.drop(["Cabin"], axis=1)

In [39]:
# drop
del_features = ["PassengerId", "Name", "Ticket", "Embarked"]
X = X.drop(del_features, axis=1)
X = X.drop(["Survived"], axis=1)
X_test = X_test.drop(del_features, axis=1)

In [40]:
# corr_mx = X.corr()
# corr_mx = X.corr(numeric_only=True)
# sns.heatmap(corr_mx, annot=True, fmt=".2f", cmap="coolwarm")

In [41]:
# scaler = MinMaxScaler()
# X[["Age", "Fare"]] = scaler.fit_transform(X[["Age", "Fare"]])
# X_test[["Age", "Fare"]] = scaler.transform(X_test[["Age", "Fare"]])

In [42]:
X.head()

,Pclass,Sex,Age,Fare,family,haveCabin
0,3,male,22.0,7.2500,1,0
1,1,female,38.0,71.2833,1,1
2,3,female,26.0,7.9250,0,0
3,1,female,35.0,53.1000,1,1
4,3,male,35.0,8.0500,0,0


## modeling

In [43]:
class Model:
    def __init__(self, model, name=None):
        self.model = model
        self.X_train, self.X_test = X, X_test
        self.y_train, self.y_test = y, y_test
        self.name = name

    def pipeline(self):
        print(f"----- {self.name if self.name else self.model.__class__.__name__} -----")
        pass

    def fit(self, X_data=None, y_data=None):
        if X_data is None:
            X_data = self.X_train
        if y_data is None:
            y_data = self.y_train
        
        print("Training...")
        self.model.fit(X_data, y_data)
        print("Train complete")

    def predict(self, data=None):
        if data is None:
            data = self.X_test
        return self.model.predict(data)

    def accuracy(self, pred=None, labels=None):
        if pred is None:
            pred = self.predict()
        if labels is None:
            labels = self.y_test
    
        acc = accuracy_score(labels, pred)
        print(f"acc: {acc:.5f}")
    
    def kfold(self):
        skf = StratifiedKFold(
            n_splits=5,
            shuffle=True,
            random_state=42
        )

        scores = []

        best_score = -1
        best_model = None
        
        for fold, (train_idx, val_idx) in enumerate(skf.split(self.X_train, self.y_train), start=1):

            print(f"-- fold {fold}/5 --")
            
            X_train = self.X_train.iloc[train_idx]
            X_val = self.X_train.iloc[val_idx]
        
            y_train = self.y_train.iloc[train_idx]
            y_val = self.y_train.iloc[val_idx]

            model = clone(self.model)
            model.fit(X_train, y_train)

            pred = model.predict(X_val)
        
            score = accuracy_score(y_val, pred)
            scores.append(score)

            if score > best_score:
                best_score = score
                best_model = model
        
            print(f"accuracy: {score:.4f}\n")

        self.model = best_model
        
        print()
        print(f"Mean accuracy: {np.mean(scores):.4f}")
        print(f"Std: {np.std(scores):.4f}")

In [44]:
cb = CatBoostClassifier(
        iterations=500,
        learning_rate=0.05,
        depth=6,
        loss_function="Logloss",
        eval_metric="Accuracy",
        random_seed=42,
        verbose=False)

In [46]:
cat_features = ["Sex"]

cb.fit(
    X,
    y,
    cat_features=cat_features
)

CatBoostClassifier(depth=6, eval_metric='Accuracy', iterations=500, learning_rate=0.05, loss_function='Logloss', random_seed=42, verbose=False)

In [47]:
cb = Model(cb)

In [49]:
cb.accuracy()

acc: 0.87321


In [50]:
cb_pred = cb.predict()

In [51]:
submission = pd.DataFrame({
    'PassengerId': ids,
    'Survived': cb_pred
})

submission.to_csv('cb_submission.csv', index=False)